In [5]:
import pandas as pd
import unicodedata


def limpiar_texto(texto):
    """Función auxiliar para quitar tildes y normalizar textos para cruces seguros"""
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().lower()
    return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')


# =========================================================
# CONFIGURACIÓN DE RUTAS
# =========================================================
BASE_DIR = r'C:\Users\USUARIO\Documents\Tesis\tesis_gfc\data'

ruta_panel_base = f'{BASE_DIR}\\final\\panel_municipio_year_enriched.csv'
ruta_panel_stata = f'{BASE_DIR}\\raw\\auxiliary\\PANEL_CARACTERISTICAS_GENERALES(2024).dta'
ruta_dane_excel = f'{BASE_DIR}\\raw\\auxiliary\\PPED-AreaMun-2018-2042_VP.xlsx'  # Ruta de tu excel del DANE
ruta_clima = f'{BASE_DIR}\\raw\\auxiliary\\clima_historico_gee.csv'
ruta_salida = f'{BASE_DIR}\\final\\dataset_consolidado_completo.csv'


# =========================================================
# 1. CARGA DE LA BASE PRINCIPAL
# =========================================================
print("1. Cargando base principal...")
df_base = pd.read_csv(ruta_panel_base)
df_base['COD_DANE'] = df_base['COD_DANE'].astype(str).str.zfill(5)


# =========================================================
# 2. INTEGRACIÓN DE POBLACIÓN (STATA + EXCEL DANE 2018-2042)
# =========================================================
print("2. Procesando e integrando población (CEDE + DANE oficial)...")

# A. Cargar datos del Stata (CEDE)
df_stata = pd.read_stata(ruta_panel_stata)
df_stata = df_stata.rename(columns={
    'codmpio': 'COD_DANE',
    'ano': 'year',
    'pobl_tot': 'poblacion_dane'
})
df_stata['COD_DANE'] = pd.to_numeric(df_stata['COD_DANE'], errors='coerce').fillna(0).astype(int).astype(str).str.zfill(5)
df_stata['year'] = pd.to_numeric(df_stata['year'], errors='coerce').fillna(0).astype(int)
df_stata_filtrado = df_stata[['COD_DANE', 'year', 'poblacion_dane']].dropna(subset=['COD_DANE', 'year'])
df_stata_filtrado = df_stata_filtrado[(df_stata_filtrado['year'] >= 2001) & (df_stata_filtrado['year'] < 2018)]

# B. Cargar datos del Excel del DANE (A partir de 2018 para asegurar 2023 en adelante)
print("   -> Leyendo proyecciones oficiales del DANE (Excel)...")
df_dane_excel = pd.read_excel(ruta_dane_excel, sheet_name='PobMunicipalxÁrea', skiprows=10)
# Las columnas del excel corresponden a: CodDpto, Depto, CodMpio, Mpio, Year, Area, Poblacion
df_dane_excel.columns = ['CodDpto', 'Depto', 'CodMpio', 'Mpio', 'Year', 'Area', 'Poblacion']

# Filtrar únicamente el total municipal
df_dane_excel = df_dane_excel[df_dane_excel['Area'].astype(str).str.strip().str.lower() == 'total'].copy()

# Estandarizar columnas
df_dane_excel['COD_DANE'] = pd.to_numeric(df_dane_excel['CodMpio'], errors='coerce').fillna(0).astype(int).astype(str).str.zfill(5)
df_dane_excel['year'] = pd.to_numeric(df_dane_excel['Year'], errors='coerce').astype(int)
df_dane_excel['poblacion_dane'] = pd.to_numeric(df_dane_excel['Poblacion'], errors='coerce')

df_dane_reciente = df_dane_excel[['COD_DANE', 'year', 'poblacion_dane']].dropna(subset=['COD_DANE', 'year'])
df_dane_reciente = df_dane_reciente[df_dane_reciente['year'] >= 2018]

# C. Unir ambas fuentes de población (Stata para histórico < 2018, Excel DANE para >= 2018)
df_poblacion_total = pd.concat([df_stata_filtrado, df_dane_reciente], ignore_index=True)
df_poblacion_total = df_poblacion_total.drop_duplicates(subset=['COD_DANE', 'year'])

# Limpiar columna vacía previa en df_base si existía
if 'poblacion_dane' in df_base.columns:
    df_base = df_base.drop(columns=['poblacion_dane'])

df_base = pd.merge(df_base, df_poblacion_total, on=['COD_DANE', 'year'], how='left')


# =========================================================
# 3. INTEGRACIÓN DE CLIMA (DESDE CLIMA_HISTORICO_GEE.CSV)
# =========================================================
print("3. Integrando datos climáticos de GEE...")
df_clima = pd.read_csv(ruta_clima)

df_clima = df_clima.rename(columns={
    'year': 'year',
    'temp_media_c': 'temp_media_c',
    'prec_anual_mm': 'prec_anual_mm',
    'ADM2_NAME': 'municipio_gee'
})

df_clima['year'] = pd.to_numeric(df_clima['year'], errors='coerce').astype(int)
df_clima['mun_limpio'] = df_clima['municipio_gee'].apply(limpiar_texto)

for col in ['temp_media_c', 'prec_anual_mm']:
    if col in df_base.columns:
        df_base = df_base.drop(columns=[col])

df_base['mun_limpio'] = df_base['NOMBRE_MPI'].apply(limpiar_texto)

df_clima_filtrado = df_clima[['mun_limpio', 'year', 'temp_media_c', 'prec_anual_mm']].drop_duplicates(subset=['mun_limpio', 'year'])

df_base = pd.merge(df_base, df_clima_filtrado, on=['mun_limpio', 'year'], how='left')
df_base = df_base.drop(columns=['mun_limpio'])


# =========================================================
# 4. ORDENAMIENTO Y EXPORTACIÓN FINAL
# =========================================================
print("4. Guardando dataset consolidado...")
df_base = df_base.sort_values(by=['COD_DANE', 'year']).reset_index(drop=True)

df_base.to_csv(ruta_salida, index=False)
print(f"¡Listo! Archivo guardado con éxito en:\n{ruta_salida}")

1. Cargando base principal...
2. Procesando e integrando población (CEDE + DANE oficial)...
   -> Leyendo proyecciones oficiales del DANE (Excel)...
3. Integrando datos climáticos de GEE...
4. Guardando dataset consolidado...
¡Listo! Archivo guardado con éxito en:
C:\Users\USUARIO\Documents\Tesis\tesis_gfc\data\final\dataset_consolidado_completo.csv
